# Westward the horizon lay flat and true as a spirit level.
-Cormac McCarthy, Blood Meridian

## Lagrange Playground
### Field Diagnostics, Recurrence, Coherence, and Trajectory Structures

This notebook walks the Lagrange Playground codebase from first terrain to diagnostic landscape.

The aim is to make structure inspectable:
to generate a field,
measure its differentiation,
trace its coherence,
test its recurrence,
embed its state-space,
and follow its motion through vector flow.

Each section builds one layer of the instrument.

#### Setup

Import the core modules and create the output directory structure.

This notebook writes figures, HTML, visualizations, and tables in a dedicated 'outputs/' folder.

In [67]:
from pathlib import Path
import numpy as np

from src.utils import ensure_output_dirs

from src.fields import(
    make_grid_2d,
    make_grid_3d,
    transformed_scalar_field,
    evolve_field,
    resonance_vector_field_3d,
)

from src.entropy import(
    local_entropy_map,
    entropy_over_time,
)

from src.recurrence import(
    recurrence_matrix,
    lag_profile,
)

from src.coherence import(
    coherence_tensor,
    structural_persistence,
    closure_coherence,
)

from src.geometry import state_space_from_history

from src.trajectories import(
    multi_seed_trajectories,
    series_from_history,
)

from src.parameter_scans import(
    scan_rule_surface,
    dataframe_to_surface,
)

from src.visualizations_3d import(
    surface_3d,
    vector_field_3d,
    trajectories_3d,
    recurrence_landscape_3d,
    coherence_topography_3d,
    state_space_cloud_3d,
    parameter_resonance_surface,
)



In [54]:
paths = ensure_output_dirs("outputs")
paths

{'base': WindowsPath('outputs'),
 'figures': WindowsPath('outputs/figures'),
 'html': WindowsPath('outputs/html'),
 'tables': WindowsPath('outputs/tables')}

### 2. Construct the Base Scalar Field

Build a 2D grid and generate the transformed scalar field that will serve as the initial terrain.

This field is the first body of the system: the surface from which later diagnostics will read entropy, coherence, and evolution.

In [55]:
X, Y = make_grid_2d(n=160, span=3.2)

F = transformed_scalar_field(
    X,
    Y,
    warp=0.9,
    twist=3.2,
)

surface_3d(
    X,
    Y,
    F,
    title="Warped Scalar Field",
    html_path=paths["html"] / "warped_scalar_field.html",
)

### 3. Local Entropy Topography

Compute a spatial entropy map over the field.

This diagnostic measures how differentiation is distributed locally across the terrain:
where the field is relatively uniform,
where it becomes varied,
and where local uncertainty gathers into visible structure.

In [56]:
H_local = local_entropy_map(
    F,
    window=9,
    bins=12,
)

surface_3d(
    X,
    Y,
    H_local,
    title="Local Entropy Topography",
    html_path=paths["html"] / "local_entropy_opography.html",
)

### 4 . Coherence Topography

Measure the fields capacity to remain intelligible to itself.

This coherence surface combines:
- local neighbor agreement, and 
- directional flow alignment

into a composite map of structural continuity.

In [57]:
agreement, flow_alignment, coherence_map = coherence_tensor(F)

coherence_topography_3d(
    X,
    Y,
    coherence_map,
    title="Coherence Topography",
    html_path=paths["html"] / "coherence_topography.html",
)

### 5. Evolve the Field Through Time

Generate a field history from the base scalar terrain.

The initial field is treated as a starting condition. The update rule then moves the field forward through time, producing a stack of states that can be inspected through entropy, recurrence, coherence, and state-space geometry.

In [58]:
history = evolve_field(
    F,
    steps=72,
    a=1.08,
    b=0.19,
)

history.shape

(73, 160, 160)

### 6. Time-Series Diagnostics

Reduce the evolving field history into diagnostic time series.

These series summarize different aspects of the system's motion:
- entropy tracks distributional differentiation over time,
- persistence tracks frame-to-frame structural continuity,
- closure tracks delayed self-relation across a fixed lag.

In [61]:
ent_series = entropy_over_time(history)
persist_series = structural_persistence(history)
closure_series = closure_coherence(history, lag=6)

ent_series.shape, persist_series.shape, closure_series.shape

((73,), (72,), (67,))

### 7. Recurrence Landscape

Build a recurrence matrix from the evolved field history.

Each time step is treated as a full system state. The recurrence matrix compares every state to every other state and marks where the system comes close to itself again.

Rendered as a 3D surface, recurrence becomes a time-by-time landscape of return.

In [62]:
R, D = recurrence_matrix(
    history,
    epsilon=0.12,
)

recurrence_landscape_3d(
    R,
    title="Recurrence Landscape",
    html_path=paths["html"] / "recurrence_landscape.html",
)

### 8. Lag Profile

Measure recurrence by temporal separation.

The lag profile reads the diagonals of the recurrence matrix and asks how strongly the system recurs at each time offset.

In [63]:
lag_values = lag_profile(R)

lag_values.shape

(73,)

In [64]:
lag_values[:10]

array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

### 9. State-Space Embedding

Embed the evolving field history into a 3D state-space.

Each field frame begins as a high-dimensional state. The embedding translates those full field states into three coordinates while preserving relational structure as much as possible.

Coloring the path by entropy lets us see whether the system moves through state-space with increasing, decreasing, or shifting differentiation.

In [65]:
state_points = state_space_from_history(
    history,
    n_components=3,
)

state_space_cloud_3d(
    state_points,
    color_values=ent_series,
    title="Entropy-Colored State-Space",
    html_path=paths["html"] / "state_space_entropy.html",
)

### 10. 3D Resonance Vector Field

Generate a 3D vector field.

This section samples directional structure across a 3D grid. The cones show how the field pushes, twists, or redirects motion at each point in space.

In [68]:
X3, Y3, Z3 = make_grid_3d(
    n=10,
    span=2.1,
)

U3, V3, W3, M3 = resonance_vector_field_3d(
    X3,
    Y3,
    Z3,
    a=1.1,
    b=0.9,
    c=0.8,
    twist=0.7,
)

vector_field_3d(
    X3,
    Y3,
    Z3,
    U3,
    V3,
    W3,
    mag=M3,
    title="Resonance Vector Field",
    html_path=paths["html"] / "resonance_vector_field.html",
)

### 11. Multi-Seed Trajectories

Trace multiple starting points through the Lagrange flow.

Each seed enters the same vector field from a different location. The resulting traces show how initial position changes the path of motion through the field.

In [69]:
seeds = np.array([
    [-1.2, -0.8,  0.6],
    [-0.6,  1.0, -0.4],
    [ 0.4, -1.2,  0.9],
    [ 1.1,  0.5, -0.8],
    [ 0.0,  0.0,  0.0],
])

traces = multi_seed_trajectories(
    seeds,
    steps=450,
    dt=0.025,
    a=1.0,
    b=0.85,
    c=0.7,
    twist=0.42,
)

trajectories_3d(
    traces,
    title="Lagrange Flow Trajectories",
    html_path=paths["html"] / "lagrange_flow_trajectories.html",
)

### 12. Parameter Resonance Surface

Scan the field evolution rule across parameter space.

Each `(a, b)` pair generates a field history. For each history, the notebook records final entropy, recurrence density, and mean structural persistence.

The result is a parameter landscape: a way to see how small changes in rule values produce different structural regimes.



**Runtime note:** This scan evaluates a full grid of parameter combinations. Each point evolves the field and computes recurrence diagnostics, so this section may take noticeably longer than the earlier visualization cells.

In [72]:
a_vals = np.linspace(0.85, 1.25, 12)
b_vals = np.linspace(0.05, 0.35, 12)

scan_df = scan_rule_surface(
    F,
    a_vals,
    b_vals,
    steps=54,
    recurrence_epsilon=0.13,
)

scan_df.head()

,a,b,final_entropy,recurrence_density,mean_persistence
0,0.85,0.050000,4.584384,0.018182,-0.004937
1,0.85,0.077273,4.584124,0.018182,-0.006165
2,0.85,0.104545,4.584293,0.018182,-0.006148
3,0.85,0.131818,4.584296,0.018182,-0.004481
4,0.85,0.159091,4.584476,0.018182,-0.005024


## 13. Render the Parameter Surface

Convert the tidy scan table into an `X, Y, Z` surface and render recurrence density as terrain.

In [73]:
Xr, Yr, Zr = dataframe_to_surface(
    scan_df,
    "a",
    "b",
    "recurrence_density",
)

parameter_resonance_surface(
    Xr,
    Yr,
    Zr,
    title="Recurrence Resonance Surface",
    html_path=paths["html"] / "recurrence_resonance_surface.html",
)

In [74]:
scan_df.to_csv(
    paths["tables"] / "parameter_scan.csv",
    index=False,
)

paths["tables"] / "parameter_scan.csv"

WindowsPath('outputs/tables/parameter_scan.csv')

# The colt stood against the horse with its head down and the horse was watching, out there past men's knowing, where the stars are drowning and whales ferry their vast souls through the black and seamless sea.
-Cormac McCarthy, Blood Meridian

### 14. Outputs

The base Lagrange Playground run is complete.

Generated artifacts include:
- warped scalar field
- local entropy topography
- coherence topography
- recurrence landscape
- entropy-colored state-space
- resonance vector field
- Lagrange flow trajectories
- recurrence resonance surface
- parameter scan table

The notebook now functions as the first complete driver for the codebase.

In [75]:
list(paths["html"].glob("*.html")), list(paths["tables"].glob("*.csv"))

([WindowsPath('outputs/html/coherence_topography.html'),
  WindowsPath('outputs/html/lagrange_flow_trajectories.html'),
  WindowsPath('outputs/html/local_entropy_opography.html'),
  WindowsPath('outputs/html/recurrence_landscape.html'),
  WindowsPath('outputs/html/recurrence_resonance_surface.html'),
  WindowsPath('outputs/html/resonance_vector_field.html'),
  WindowsPath('outputs/html/state_space_entropy.html'),
  WindowsPath('outputs/html/warped_scalar_field.html')],
 [WindowsPath('outputs/tables/parameter_scan.csv')])